# Recursive subagents and scope

In tutorial 1 every input was a variable in *one* agent's REPL. But an agent can call `invoke` itself, making a tree of agents — which is how you split a job too big for one REPL session.

That raises the obvious question: what can a **subagent** see? The answer is the point of this tutorial, and it is not “everything”.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [ ]:
%pip install -q jaz-lang pandas

In [1]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [2]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## An agent calling agents

`invoke` is already available inside the REPL — nothing to import or pass in. Here the
top-level agent farms each species out to its own subagent.

In [3]:
import math

import pandas as pd

from jaz import invoke

# The same data and tool as tutorial 1.
PENGUINS = (
    "https://raw.githubusercontent.com"
    "/mwaskom/seaborn-data/master/penguins.csv"
)
penguins = pd.read_csv(PENGUINS)


def same_distribution_p_value(a: list[float], b: list[float]) -> float:
    """Mann-Whitney U test: the p-value that two samples come from the same
    distribution. Missing values (NaN) are ignored."""
    a, b = [x for x in a if x == x], [y for y in b if y == y]
    n1, n2 = len(a), len(b)
    u = sum((x > y) + (x == y) / 2 for x in a for y in b)
    z = (abs(u - n1 * n2 / 2) - 0.5) / math.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    return math.erfc(z / math.sqrt(2))


STATS_TASK = (
    "For each species s in species, call invoke(...) to have a subagent compute the mean "
    "body_mass_g of species s in penguins. Give each subagent the task "
    "f'Return the mean body_mass_g of {s} penguins as a float.' and the inputs it needs. "
    "Return a dict mapping each species to its mean."
)
SPECIES = ["Adelie", "Chinstrap", "Gentoo"]

stats = invoke(
    task=STATS_TASK,
    species=SPECIES,
    penguins=penguins,
)

stats

{'Adelie': 3700.662251655629,
 'Chinstrap': 3733.0882352941176,
 'Gentoo': 5076.016260162602}

## Inputs stop at the agent that received them

Here is the part that surprises people. An input you pass to `invoke` is bound in **that agent's** REPL and nowhere else. Subagents do not inherit it.

We check it by reading the prompts rather than asking an agent to tell us — the prompt is ground truth. (Same scaffolding as last time; skip the class.)

In [4]:
from jaz.hooks import Hook


class CollectPrompts(Hook):
    """Records the prompt of every invoke in the tree.

    Same scaffolding as tutorial 1's Logger, just keeping the prompts instead of printing
    them. You do not need to understand it; hooks are tutorial 3.
    """

    def __init__(self):
        self.prompts = []

    def on_llm_query_enter(self, event):
        whole = "\n".join(m["content"] for m in event.messages)
        if whole not in self.prompts:
            self.prompts.append(whole)
        return []


TASK = (
    "Call invoke(task='Compute the mean body_mass_g of penguins.', penguins=penguins) "
    "exactly as written, with no other arguments, and return its result."
)

In [5]:
seen = CollectPrompts()

with seen:
    invoke(task=TASK, penguins=penguins, same_distribution_p_value=same_distribution_p_value)

for i, prompt in enumerate(seen.prompts):
    print(f"invoke #{i}  can see same_distribution_p_value: {'same_distribution_p_value(' in prompt}")

invoke #0  can see same_distribution_p_value: True
invoke #1  can see same_distribution_p_value: False


The top-level agent can see `same_distribution_p_value`. The subagent it spawned cannot — the name is not bound down there at all.

That is deliberate: an agent's inputs are its own, and a deep tree does not silently accumulate every caller's variables. But it does mean that passing a tool the ordinary way gives it to exactly one agent.

## `scope` — inputs for a whole subtree

`scope` binds values as **ambient inputs**: everything inside the `with` block sees them, nested invokes included, all the way down.

Same code, one line different:

In [6]:
from jaz import scope

seen = CollectPrompts()

with seen, scope(same_distribution_p_value=same_distribution_p_value):
    invoke(task=TASK, penguins=penguins)

for i, prompt in enumerate(seen.prompts):
    print(f"invoke #{i}  can see same_distribution_p_value: {'same_distribution_p_value(' in prompt}")

invoke #0  can see same_distribution_p_value: True
invoke #1  can see same_distribution_p_value: True


Both can see it now, and nothing else changed.

This is the usual way to hand an agent its tools: most tools should be available to the whole tree rather than to the top call alone. It is equivalent to passing the value to every `invoke` beneath it by hand, without the hand-threading.

(A detail you may notice if you print the prompts: a per-call input is described in the **user** message, a scoped value in the **system** message — it is ambient, so it belongs to the session rather than to one request.)

Now the first example again, with the data in scope instead of passed to the top-level agent:

In [7]:
with scope(penguins=penguins):
    stats = invoke(task=STATS_TASK, species=SPECIES)

stats

{'Adelie': 3700.662251655629,
 'Chinstrap': 3733.0882352941176,
 'Gentoo': 5076.016260162602}

The same result. In the first run the top-level agent had to pass `penguins` down to each of
its subagents itself, because its inputs stopped at its own REPL. Here there is nothing to pass:
every subagent already sees it.

## What to take away

- **A subagent is just `invoke` called from inside the REPL.**
- **Inputs do not propagate.** They belong to the agent that received them.
- **`scope` reaches the subtree.** Bind tools and shared data there, not as per-call inputs.

Recursion is unbounded by default. Capping it — like everything else that shapes a run — is done with a hook, which is next.

---

**Next:** [Hooks, events and effects](03-hooks-events-and-effects.ipynb) — including what those `Hook` classes you have been skipping actually do.